In [2]:
%pip install --upgrade pip
%pip install torch torchvision matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms # type: ignore
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [7]:
batch_size = 64
lr = 0.0002
epochs = 10
latent_dim = 100

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.96MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 260kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.13MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.08MB/s]


In [17]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 28*28),
            nn.Tanh()
        )
    def forward(self, z):
        return self.model(z)

    

In [18]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

In [19]:
G = Generator().to(device)
D = Discriminator().to(device)

In [20]:
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr)
optimizer_D = optim.Adam(D.parameters(), lr=lr)

In [21]:
def show_images(images, epoch):
    images = images.view(-1, 1, 28, 28).cpu().data
    grid = torchvision.utils.make_grid(images, nrow=8, normalize=True) # type: ignore
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"Epoch {epoch}")
    plt.axis("off")
    plt.show()


In [24]:
for epoch in range(epochs):
    for batch_idx, (real, _) in enumerate(dataloader):

        real = real.view(-1, 784).to(device)
        batch_size_curr = real.size(0)

        # Labels
        real_labels = torch.ones(batch_size_curr, 1).to(device)
        fake_labels = torch.zeros(batch_size_curr, 1).to(device)
    
        z = torch.randn(batch_size_curr, latent_dim).to(device)
        fake = G(z)

        D_real = D(real)
        D_fake = D(fake.detach())

        loss_real = criterion(D_real, real_labels)
        loss_fake = criterion(D_fake, fake_labels)

        loss_D = loss_real + loss_fake

        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()
        z = torch.randn(batch_size_curr, latent_dim).to(device)
        fake = G(z)

        output = D(fake)
        loss_G = criterion(output, real_labels)

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {loss_D:.4f}, Loss G: {loss_G:.4f}")

    # Show generated images
    z = torch.randn(16, latent_dim).to(device)
    fake_images = G(z)
    show_images(fake_images, epoch)

Epoch [1/10] | Loss D: 0.0035, Loss G: 8.0229


NameError: name 'torchvision' is not defined